<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 El Álbum con Fotos Rotas y las Páginas que Faltan 🧩
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Machine Learning
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 08
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Machine%20Learning/08%20-%20Temas%20Avanzados%20de%20ML/Para%20Dummies/04_Datos_Faltantes_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **«para no ingenieros»** del [cuaderno 04 del Módulo 08 (Datos faltantes)](../04_Datos_Faltantes.ipynb). Casi ningún conjunto de datos viene completo: encuestas con preguntas sin responder, sensores que fallan. ¿Qué haces con los **huecos**? Depende de **por qué faltan**, y **rellenar sin pensar** puede engañarte. Solo usamos *NumPy* y gráficas.

Al terminar podrás explicar:
1. Las **tres historias** de por qué faltan datos (**al azar**, **por otra respuesta**, **por la respuesta misma**) y por qué solo una es inofensiva.
2. Por qué **rellenar con el promedio** hace que **todos parezcan iguales** y esconde la variedad real.
3. Por qué **preguntar a los más parecidos** (vecinos) rellena mejor.
4. Por qué **tirar las encuestas incompletas** desperdicia casi todo.

> 🔗 Si luego quieres ver la versión con más detalle: [cuaderno estándar](../04_Datos_Faltantes.ipynb). Antes de esto, [la aguja en el pajar (clases raras)](03_Desbalanceados_Dummies.ipynb).

---
## 1. ¿Por qué faltan los datos? Tres historias 📋

Una encuesta pregunta el **sueldo** a 2000 personas. Pero **30 %** no responde. Según **por qué** no respondieron, el sueldo promedio de **quienes sí respondieron** se parece o no al real:

| Historia | Por qué falta | Ejemplo | ¿Se puede confiar en el promedio de quienes respondieron? |
|---|---|---|---|
| 🎲 **Al azar** (MCAR) | Un fallo sin relación con nada | Se perdieron hojas en el correo | ✅ Sí (hay menos datos, pero sin sesgo) |
| 🔗 **Por otra respuesta** (MAR) | Depende de **algo que sí sabemos** | Las personas **mayores** (edad conocida) responden menos, y las mayores ganan más | ⚠️ **No**, pero **se puede corregir usando la edad** |
| 🤐 **Por la respuesta misma** (MNAR) | Depende **del propio sueldo** | **Quien gana mucho** no quiere decirlo | ❌ **No**, y **no hay forma de corregirlo** con estos datos |

**Dato importante:** con solo los datos **no se puede saber** cuál historia es la verdadera; hay que **conocer cómo se recogieron los datos**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(1)
n = 2000
edad = rng.normal(0, 1, n)                                              # edad (estandarizada), siempre conocida
sueldo = 0.7 * edad + np.sqrt(1 - 0.7 ** 2) * rng.normal(0, 1, n)       # el sueldo crece con la edad (estandarizado: promedio real = 0)

def sin_responder(score, objetivo=0.30, fuerza=1.5):
    """Probabilidad de NO responder que crece con 'score', ajustada para que ~30 % no responda."""
    lo, hi = -10, 10
    for _ in range(60):
        a = (lo + hi) / 2
        if (1 / (1 + np.exp(-(a + fuerza * score)))).mean() > objetivo: hi = a
        else: lo = a
    return 1 / (1 + np.exp(-(a + fuerza * score)))

historias = {"Al azar": np.full(n, 0.30), "Por la edad (conocida)": sin_responder(edad), "Por el propio sueldo": sin_responder(sueldo)}
promedios = {nombre: [] for nombre in historias}
corregidos = {nombre: [] for nombre in historias}
for _ in range(300):                                                    # repetimos la encuesta 300 veces para ver el resultado "tipico"
    edad_ = rng.normal(0, 1, n); sueldo_ = 0.7 * edad_ + np.sqrt(1 - 0.7 ** 2) * rng.normal(0, 1, n)
    probs = {"Al azar": np.full(n, 0.30), "Por la edad (conocida)": sin_responder(edad_), "Por el propio sueldo": sin_responder(sueldo_)}
    for nombre, p in probs.items():
        no_responde = rng.random(n) < p
        responde = ~no_responde
        promedios[nombre].append(sueldo_[responde].mean())
        pendiente, origen = np.polyfit(edad_[responde], sueldo_[responde], 1)         # usamos la edad (conocida de TODOS) para corregir
        corregidos[nombre].append(np.mean(origen + pendiente * edad_))

print(f"{'Historia':<30}{'Promedio de quienes respondieron':>34}{'Corregido usando la edad':>28}   (el real es 0.00)")
for nombre in historias:
    print(f"{nombre:<30}{np.mean(promedios[nombre]):>34.2f}{np.mean(corregidos[nombre]):>28.2f}")

fig, ax = plt.subplots(figsize=(9, 4.4))
pos = np.arange(3)
ax.bar(pos - 0.18, [np.mean(promedios[h]) for h in historias], 0.36, color="#dc2626", label="Promedio de quienes respondieron")
ax.bar(pos + 0.18, [np.mean(corregidos[h]) for h in historias], 0.36, color="#16a34a", label="Corregido usando la edad")
ax.axhline(0, color="black", lw=1.2, label="Sueldo promedio real")
ax.set_xticks(pos); ax.set_xticklabels(list(historias), fontsize=9); ax.set_ylabel("Sueldo promedio estimado (estandarizado)"); ax.legend(fontsize=8)
ax.set_title("Solo cuando falta al azar o por algo conocido se puede confiar (o corregir)", fontweight="bold", fontsize=10)
plt.tight_layout(); plt.show()
m = {h: np.mean(promedios[h]) for h in historias}; c = {h: np.mean(corregidos[h]) for h in historias}
assert abs(m["Al azar"]) < 0.03 and m["Por la edad (conocida)"] < -0.15 and m["Por el propio sueldo"] < m["Por la edad (conocida)"]
assert abs(c["Por la edad (conocida)"]) < 0.03 and c["Por el propio sueldo"] < -0.1

---
---
**Lección:** si falta **al azar**, el promedio de quienes respondieron **sirve**. Si falta por la **edad** (que conocemos), el promedio queda **bajo** (los mayores, que ganan más, responden menos) pero **usando la edad se corrige por completo**. Si falta por **el propio sueldo**, el promedio queda **muy bajo** y **ni usando la edad se arregla**: hacen falta **datos externos** o **supuestos explícitos**. Antes de rellenar, **pregúntate por qué faltan**.

## 2. Rellenar con el promedio vs. preguntar a los vecinos 👥

Tenemos la **estatura** y el **peso** de 500 personas (están **relacionados**: los más altos suelen pesar más). A 30 % le **borramos el peso**. Dos formas de rellenar:

- 📊 **El promedio:** a todos los que no dieron su peso les ponemos **el peso promedio de todos**. Es rápido, pero **trata igual a un adulto alto y a uno bajo**.
- 👥 **Preguntar a los vecinos:** buscamos a las **5 personas con estatura más parecida** y usamos **su peso promedio**.

Como **sabemos el peso real** de los que borramos, podemos **medir el error** de cada método. Y miramos qué le pasa a la **variedad** de los pesos.

In [ ]:
rng = np.random.default_rng(2)
n = 500
estatura = rng.normal(170, 9, n)
peso = 70 + 0.9 * (estatura - 170) + rng.normal(0, 6, n)                 # a más estatura, más peso (con variación individual)

falta = rng.random(n) < 0.30                                             # le borramos el peso al 30 %
peso_obs = np.where(falta, np.nan, peso)

# 1) Rellenar con el promedio
promedio = np.nanmean(peso_obs)
peso_prom = np.where(falta, promedio, peso)

# 2) Rellenar preguntando a las 5 personas con estatura mas parecida (que SI dieron su peso)
con_peso = np.where(~falta)[0]
peso_vec = peso_obs.copy()
for i in np.where(falta)[0]:
    cercanos = con_peso[np.argsort(np.abs(estatura[con_peso] - estatura[i]))[:5]]
    peso_vec[i] = peso[cercanos].mean()

error = lambda relleno: np.sqrt(np.mean((relleno[falta] - peso[falta]) ** 2))
print(f"Error al adivinar los pesos borrados (en kg):  promedio = {error(peso_prom):.1f} | vecinos = {error(peso_vec):.1f}")
print(f"Variedad de los pesos (desviación típica): real = {peso.std():.1f} | rellenado con el promedio = {peso_prom.std():.1f} | rellenado con vecinos = {peso_vec.std():.1f}")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.4), sharey=True)
for ax, (nombre, relleno, c) in zip(axes, [("Realidad", peso, "#2563eb"), ("Rellenado con el PROMEDIO", peso_prom, "#dc2626"), ("Rellenado con VECINOS", peso_vec, "#16a34a")]):
    ax.scatter(estatura[~falta], relleno[~falta], s=10, color="#94a3b8", label="dieron su peso")
    ax.scatter(estatura[falta], relleno[falta], s=14, color=c, label="peso rellenado" if nombre != "Realidad" else "(los que no lo dieron)")
    ax.set_xlabel("Estatura (cm)"); ax.set_title(nombre, fontweight="bold", fontsize=10); ax.legend(fontsize=8)
axes[0].set_ylabel("Peso (kg)")
plt.tight_layout(); plt.show()
assert error(peso_vec) < 0.85 * error(peso_prom)                            # con variables relacionadas, los vecinos aciertan mas
assert peso_prom.std() < 0.95 * peso.std()                                  # el promedio encoge la variedad

---
---
**Lección:** el **promedio** pone a todos los datos rellenados en **una línea plana** (un único peso): **encoge la variedad** y **borra la relación** entre estatura y peso. Los **vecinos** respetan que los más altos pesan más y **adivinan mejor**. Pero esta ventaja existe **porque las variables están relacionadas**: si la estatura no dijera nada del peso, los vecinos **no ayudarían** más que el promedio.

## 3. Tirar las encuestas incompletas y la regla de no hacer trampa 🗑️

Una opción «fácil» es **tirar** toda encuesta que tenga **algún** hueco. Parece inocente… Si cada pregunta se queda sin responder **5 %** de las veces (poco), y hay **20 preguntas**, ¿cuántas encuestas llegan **completas**? Cada pregunta se responde con probabilidad 0.95; las 20 a la vez: 0.95 multiplicado **20 veces**.

In [ ]:
rng = np.random.default_rng(3)
encuestas, preguntas, p_hueco = 5000, 20, 0.05
huecos = rng.random((encuestas, preguntas)) < p_hueco
completas = (~huecos.any(axis=1)).mean()
teoria = (1 - p_hueco) ** preguntas
print(f"Encuestas completas: {completas:.1%}  (cálculo: 0.95 multiplicado 20 veces = {teoria:.1%}). ¡Tirar las incompletas desperdicia ~2 de cada 3 encuestas!")

preg = np.arange(1, 41)
fig, ax = plt.subplots(figsize=(8.5, 4.2))
for p, c in [(0.01, "#16a34a"), (0.05, "#f59e0b"), (0.10, "#dc2626")]:
    ax.plot(preg, 100 * (1 - p) ** preg, lw=2, color=c, label=f"{int(100 * p)} % sin responder por pregunta")
ax.axvline(20, color="black", ls=":"); ax.set_xlabel("Número de preguntas"); ax.set_ylabel("% de encuestas completas"); ax.legend(fontsize=8)
ax.set_title("Con muchas preguntas, casi ninguna encuesta llega completa", fontweight="bold", fontsize=10)
plt.tight_layout(); plt.show()
assert abs(completas - teoria) < 0.02 and completas < 0.4

---
---
**Lección:** con 20 preguntas y solo 5 % de huecos, **queda apenas ~36 %** de las encuestas: tirar encuestas incompletas **desperdicia la mayoría** de los datos. Es mejor **rellenar con cuidado**.

**Y una regla de oro para no hacerte trampa:** para **rellenar** los huecos, usa **solo** la información de la **parte de práctica** (el material con el que estudias), **nunca** del examen final, y **jamás** las respuestas del examen (si rellenas con «el promedio de los que aprobaron», **le regalas la respuesta al modelo**). En los programas esto se hace metiendo el rellenado **dentro del mismo «paquete de pasos»** (*pipeline*) que el modelo.

## Resumen en una frase 🎯

> **Antes de rellenar un hueco, pregúntate por qué falta (al azar, por otra respuesta o por la respuesta misma); rellenar con el promedio aplana y engaña, preguntar a los parecidos rellena mejor si las cosas están relacionadas, tirar encuestas incompletas desperdicia casi todo, y todo el relleno se aprende solo con la parte de práctica.**

### Lo que aprendiste hoy:
- ✅ **Tres historias**: falta al azar (inofensivo), por otra respuesta (**se corrige** usándola) o por la respuesta misma (**no se corrige** con estos datos).
- ✅ **El promedio** hace que todos los rellenados sean iguales: **encoge la variedad** y **borra relaciones**.
- ✅ **Los vecinos** (los más parecidos) adivinan mejor **si las variables están relacionadas**.
- ✅ **Tirar encuestas incompletas** deja ~36 % con 20 preguntas y 5 % de huecos: **desperdicia** casi todo.
- ✅ **Regla de oro:** rellenar con información **solo de la práctica**, nunca del examen final ni de las respuestas.

> 🎓 **Siguiente paso:** el [cuaderno estándar](../04_Datos_Faltantes.ipynb) implementa el relleno por vecinos **desde cero** (comparado con `scikit-learn`), mide el error por cada historia, y explica la **imputación múltiple**. Con esto cierras el Módulo 08; el siguiente es el **Módulo 09** (casos de estudio, imágenes, lenguaje y recomendaciones).

---
##### 🛠️ Práctica 1: ¿A cuántos vecinos preguntar? Probar distintos valores de $k$

En la Sección 2 se rellenó el peso preguntando a las **5** personas con estatura más parecida. ¿Y si se pregunta a **1**, a **15** o a **300**? Usa `estatura`, `peso`, `peso_obs`, `falta`, `con_peso`, `peso_prom` y la función `error(relleno)`:

1. Escribe una función `rellenar_con_vecinos(k)`: para cada persona sin peso, toma las `k` personas con estatura más parecida (de las que sí dieron su peso) y usa el promedio de **su** peso. Devuelve el arreglo ya relleno.
2. Calcula `errores_k`, un diccionario con el error (usando `error(...)`) para `k` en `[1, 5, 15, 50, 300]`.
3. Imprime los errores junto al del promedio, `error(peso_prom)`.
4. Comprueba con `assert` que preguntar a 5 vecinos es mejor que preguntar a uno solo, que con 15 vecinos el error es menos del 75 % del que da el promedio, y que con 300 vecinos el error vuelve a empeorar.

In [ ]:
# Escribe tu código aquí

# def rellenar_con_vecinos(k):
#     relleno = peso_obs.copy()
#     for i in np.where(falta)[0]:
#         cercanos = con_peso[np.argsort(np.abs(estatura[con_peso] - estatura[i]))[:...]]
#         relleno[i] = peso[cercanos].mean()
#     return relleno
#
# errores_k = {k: error(rellenar_con_vecinos(k)) for k in [1, 5, 15, 50, 300]}


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
def rellenar_con_vecinos(k):
    relleno = peso_obs.copy()
    for i in np.where(falta)[0]:                                                      # cada persona sin peso
        cercanos = con_peso[np.argsort(np.abs(estatura[con_peso] - estatura[i]))[:k]]  # las k de estatura mas parecida
        relleno[i] = peso[cercanos].mean()                                            # su peso promedio
    return relleno

errores_k = {k: error(rellenar_con_vecinos(k)) for k in [1, 5, 15, 50, 300]}
for k, e in errores_k.items():
    print(f"k = {k:>3}: error = {e:.1f} kg")
print(f"Rellenar con el promedio de todos: error = {error(peso_prom):.1f} kg")

assert errores_k[5] < errores_k[1]
assert errores_k[15] < 0.75 * error(peso_prom)
assert errores_k[300] > errores_k[15]
print("Con k = 1 el relleno es muy ruidoso; con k enorme se parece al promedio; en el medio está lo mejor.")
```
</details>

---


---
### 🧠 ¿Ya lo tienes claro?

1. En una encuesta de salud, las personas con **peor salud** no responden la pregunta sobre su salud. ¿Cuál de las tres historias es? ¿Se puede corregir? ¿Por qué es la más difícil?
2. ¿Por qué rellenar con el promedio hace que dos personas muy distintas (una alta y otra baja) tengan el **mismo** peso rellenado? ¿Qué daño hace eso al estudiar la relación entre estatura y peso?
3. Un amigo dice: «relleno los huecos con el promedio de todo el conjunto, incluyendo el examen final, y luego entreno». ¿Qué trampa hay y cómo la evitarías?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Machine Learning (Edición Didáctica Para Dummies)</i>
  </p>
</div>